# 03. Baseline Modeling

This notebook establishes a reproducible baseline for the UCI Bank Marketing classification task.

The data understanding and preprocessing decisions were established in the previous notebooks. We now focus on training and evaluating a baseline model that will serve as the reference point for future experiments.

## 1. Introduction

### 1.1 Objective & Scope

The target variable is `y`:

- `1` (`yes`): the customer subscribed to a term deposit.
- `0` (`no`): the customer did not subscribe.

Predictions are made immediately before a planned marketing contact.

The business objective is to prioritize customers with higher predicted subscription probabilities under limited call-center capacity.

This notebook aims to:

- Reproduce the established chronological data split and selected preprocessing configuration.
- Train a Logistic Regression baseline.
- Evaluate its performance on the validation set.
- Establish reference metrics for subsequent model experiments.

Model comparison, hyperparameter optimization, and final test evaluation are outside the scope of this notebook.

### 1.2 Baseline Strategy

We use Logistic Regression as the initial baseline model.

Logistic Regression provides a relatively simple and interpretable reference model and supports probability-based ranking through `predict_proba()`.

The selected preprocessing strategy from `02-preprocessing.ipynb` is preserved:

- Exclude `duration` to prevent prediction-time leakage.
- Retain `campaign` and `contact` under the established feature availability assumptions.
- Standardize numeric features, including raw `pdays`.
- Apply one-hot encoding to categorical features.

Preprocessing and model training will be combined using a scikit-learn `Pipeline`.

The established chronological split is also preserved:

- Training: first 70%.
- Validation: next 20%.
- Test: final 10%.

The test set remains frozen for final evaluation.

### 1.3 Evaluation Metrics

Because the business objective emphasizes customer prioritization rather than hard classification, evaluation focuses on probability ranking.

**Primary metric**

- Average Precision (AP)

**Secondary metric**

- ROC-AUC

**Business-oriented ranking metrics**

- Precision@K
- Recall@K
- Lift@K

We will initially examine ranking performance at contact capacities of 5%, 10%, and 20%.

These capacities are illustrative evaluation points rather than finalized contact policies.

Accuracy is not used as the primary model-selection metric because the dataset is imbalanced and the business objective is ranking-oriented.

The resulting validation metrics will serve as reference values for model comparison and improvement in `04-model-experiments.ipynb`.


In [2]:
target = "y"
positive_label = "yes"

excluded_features = ["duration"]

primary_metric = "average_precision"


## 2. Data & Pipeline Setup

This section prepares the data, reproduces the established chronological split, and reconstructs the selected preprocessing and baseline model configurations.

### 2.1 Load Data & Reproduce Chronological Split

We load the original UCI Bank Marketing dataset and reproduce the chronological split established during EDA and preprocessing.

The dataset is divided according to its original row order:

- Training: first 70%
- Validation: next 20%
- Test: final 10%

Random or stratified splitting is not used because the dataset contains temporal structure and substantial distribution shifts.

The training set will be used to fit preprocessing steps and the baseline model.

The validation set will be used to evaluate baseline performance.

The test set is retained but remains frozen for final evaluation.


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

In [4]:
PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT

PosixPath('/home/moonk/projects/bank-marketing-ml')

In [5]:
RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bank-additional"
    / "bank-additional-full.csv"
)

RAW_DATA_PATH

PosixPath('/home/moonk/projects/bank-marketing-ml/data/raw/bank-additional/bank-additional-full.csv')

In [6]:
df = pd.read_csv(RAW_DATA_PATH, sep=";")

In [7]:
n_rows = len(df)

train_end = int(n_rows * 0.7)
valid_end = int(n_rows * 0.9)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

train_df.shape, valid_df.shape, test_df.shape


((28831, 21), (8238, 21), (4119, 21))

### 2.2 Define Features & Target

We separate the modeling features and target variable using the decisions established during EDA and preprocessing.

The target variable `y` is converted into binary labels:

- `no` → 0
- `yes` → 1

The `duration` feature is excluded because it is unavailable immediately before the planned marketing contact and would introduce target leakage.

The remaining 19 predictors are retained as modeling features.

We prepare the training and validation datasets while keeping the original split dataframes unchanged.

The test set remains frozen for final evaluation.


In [8]:
modeling_features = [
    col
    for col in df.columns
    if col not in [target, *excluded_features]
]

X_train = train_df[modeling_features].copy()
X_valid = valid_df[modeling_features].copy()

y_train = train_df[target].map({"no": 0, "yes": 1})
y_valid = valid_df[target].map({"no": 0, "yes": 1})

print(f"Modeling features: {len(modeling_features)}")
print(f"X_train: {X_train.shape}")
print(f"X_valid: {X_valid.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_valid: {y_valid.shape}")


Modeling features: 19
X_train: (28831, 19)
X_valid: (8238, 19)
y_train: (28831,)
y_valid: (8238,)


### 2.3 Rebuild Selected Preprocessing Pipeline

We reconstruct the baseline preprocessing configuration selected in `02-preprocessing.ipynb`.

The selected preprocessing strategy is:

- **Numeric features:** Apply `StandardScaler` to numeric features, including raw `pdays`.
- **Categorical features:** Apply `OneHotEncoder(handle_unknown="ignore")`.
- **Other columns:** Exclude them using `remainder="drop"`.

The `"unknown"` category is retained, and rare categories are not manually grouped.

The raw `pdays` representation is preserved because it achieved slightly better validation performance than the sentinel-aware representation in the previous Logistic Regression experiment.

The preprocessing configuration is implemented using scikit-learn `ColumnTransformer`.

At this stage, the preprocessor is only defined. It will be fitted on the training data as part of the complete modeling pipeline.


In [12]:
numeric_features = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

raw_numeric_features = numeric_features + ["pdays"]


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            raw_numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ],
    remainder="drop",
)

preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

### 2.4 Define Baseline Model

We use Logistic Regression as the initial baseline classifier.

Logistic Regression provides a simple and interpretable reference model for subsequent experiments.

It also supports probability-based customer ranking through `predict_proba()`, which aligns with the business objective of prioritizing customers with higher subscription probabilities.

To reproduce the previous preprocessing experiment, we retain the same model configuration:

- `LogisticRegression(max_iter=1000)`
- Default regularization and other hyperparameters

No hyperparameter tuning is performed at this stage.

The model will be combined with the selected preprocessing configuration and fitted on the training data in the next section.


In [14]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000)

baseline_model


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

## 3. Baseline Training

This section builds the complete baseline pipeline, fits it on the training data, and generates predicted probabilities for validation.

### 3.1 Build Model Pipeline

We combine the selected preprocessing configuration and Logistic Regression into a single scikit-learn `Pipeline`.

The pipeline contains two steps:

1. **Preprocessor:** Apply the selected numeric scaling and categorical encoding.
2. **Model:** Fit the Logistic Regression classifier on the transformed features.

Using a pipeline ensures that preprocessing and model training are managed together.

All learned preprocessing parameters will be fitted on the training data only, preventing validation data from influencing the training process.


In [15]:
from sklearn.pipeline import Pipeline

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", baseline_model),
    ]
)

baseline_pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the

### 3.2 Train Model

We fit the complete Logistic Regression baseline pipeline using the training data.

Calling `fit()` performs two operations:

1. Fit the preprocessing transformations on `X_train`.
2. Train Logistic Regression using the transformed features and `y_train`.

All learned preprocessing parameters and model coefficients are estimated exclusively from the training data.

The validation set is reserved for evaluating the trained baseline, while the test set remains frozen for final evaluation.

No hyperparameter tuning or model selection is performed at this stage.


In [16]:
baseline_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['age','job','marital',...,'cons.conf.idx','euribor3m','nr.employed']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all re

### 3.3 Generate Validation Predictions

We generate predicted subscription probabilities for the validation set using the fitted Logistic Regression baseline pipeline.

Since the business objective is customer prioritization, we use `predict_proba()` rather than hard class predictions.

The positive class (`y = 1`) represents customers who subscribed to a term deposit.

The fitted pipeline automatically applies the preprocessing transformations learned from the training data before generating predictions.

No preprocessing or model fitting is performed on the validation set.

These predicted probabilities will be used to calculate validation ranking metrics in the next section.


In [17]:
# Confirm the class order
baseline_pipeline.classes_


array([0, 1])

In [18]:
y_valid_proba = baseline_pipeline.predict_proba(X_valid)[:, 1]

y_valid_proba[:10]


array([0.14032551, 0.12653996, 0.10324642, 0.15244576, 0.1793063 ,
       0.16981054, 0.13633531, 0.19365651, 0.13333615, 0.26930728])

## 4. Baseline Evaluation

This section evaluates the Logistic Regression baseline using the chronological validation set.

The evaluation focuses on probability ranking rather than hard class predictions.

### 4.1 General Ranking Metrics

We evaluate the baseline using two threshold-independent ranking metrics:

- **Average Precision (AP):** Summarizes the precision-recall relationship across ranking thresholds. This is our primary evaluation metric.
- **ROC-AUC:** Measures how well the model ranks positive observations above negative observations.

Both metrics are computed using the predicted positive-class probabilities.

We also compare the results with the previous Logistic Regression experiment in `02-preprocessing.ipynb` to verify reproducibility.

No model fitting or hyperparameter tuning is performed during evaluation.


In [19]:
from sklearn.metrics import average_precision_score, roc_auc_score

baseline_ap = average_precision_score(
    y_valid,
    y_valid_proba,
)

baseline_roc_auc = roc_auc_score(
    y_valid,
    y_valid_proba,
)

print(f"Validation AP      : {baseline_ap:.6f}")
print(f"Validation ROC-AUC : {baseline_roc_auc:.6f}")


Validation AP      : 0.297231
Validation ROC-AUC : 0.692923


### 4.2 Business-Oriented Ranking Metrics

The business objective is to prioritize customers with higher predicted subscription probabilities under limited call-center capacity.

To evaluate the practical usefulness of the baseline ranking, we examine three contact-capacity scenarios:

- Top 5% of validation customers
- Top 10% of validation customers
- Top 20% of validation customers

Customers are ranked in descending order of predicted subscription probability.

We calculate the following metrics:

- **Precision@K:** Proportion of actual subscribers among the top K customers.
- **Recall@K:** Proportion of all actual subscribers captured within the top K customers.
- **Lift@K:** Precision@K divided by the overall validation subscription rate.

Lift@K measures how strongly the model concentrates actual subscribers relative to the validation population's base rate.

These contact capacities are illustrative and are not used to select a final ranking policy.

All metrics are computed on the validation set using the fitted Logistic Regression baseline.


In [24]:
# Rank customers by predicted subscription probability
ranked_indices = np.argsort(
    -y_valid_proba,
    kind="stable",
)

ranked_y = y_valid.to_numpy()[ranked_indices]

# Cumulative number of actual subscribers
cumulative_positives = np.cumsum(ranked_y)

total_positives = y_valid.sum()
base_rate = y_valid.mean()

# Evaluate different contact capacities
top_fractions = [0.05, 0.10, 0.20]

results = []

for fraction in top_fractions:
    k = int(np.ceil(len(y_valid) * fraction))

    captured = cumulative_positives[k - 1]

    precision_at_k = captured / k
    recall_at_k = captured / total_positives
    lift_at_k = precision_at_k / base_rate

    results.append(
        {
            "Top %": f"{fraction:.0%}",
            "K": k,
            "Captured": captured,
            "Precision@K": precision_at_k,
            "Recall@K": recall_at_k,
            "Lift@K": lift_at_k,
        }
    )

ranking_results = pd.DataFrame(results)

ranking_results.round(4)


,Top %,K,Captured,Precision@K,Recall@K,Lift@K
0,5%,412,177,0.4296,0.1550,3.0991
1,10%,824,318,0.3859,0.2785,2.7839
2,20%,1648,503,0.3052,0.4405,2.2017


### 4.3 Baseline Performance Analysis

The Logistic Regression baseline was evaluated on the chronological validation set using both general and business-oriented ranking metrics.

**General ranking performance**

- Average Precision: 0.297231
- ROC-AUC: 0.692923
- Validation positive rate: approximately 13.86%

The baseline demonstrates ranking ability above a random-ranking reference, although there is room for improvement.

**Business-oriented ranking performance**

| Contact Capacity | Customers | Captured Subscribers | Precision@K | Recall@K | Lift@K |
|---|---:|---:|---:|---:|---:|
| Top 5% | 412 | 177 | 0.4296 | 0.1550 | 3.0991 |
| Top 10% | 824 | 318 | 0.3859 | 0.2785 | 2.7839 |
| Top 20% | 1,648 | 503 | 0.3052 | 0.4405 | 2.2017 |

**Key observations**

1. The baseline concentrates actual subscribers among higher-ranked customers. At the top 5%, the observed subscription rate is approximately 3.10 times the overall validation positive rate.

2. Expanding contact capacity from 5% to 20% increases the proportion of actual subscribers captured from 15.50% to 44.05%.

3. In the evaluated scenarios, Precision@K and Lift@K decrease as contact capacity increases, while Recall@K increases.

4. The results illustrate a trade-off between the proportion of subscribers among contacted customers and the proportion of all subscribers captured.

**Interpretation and limitations**

The baseline provides a useful reference for customer prioritization, but these results are specific to the chronological validation period.

The dataset exhibits substantial temporal distribution shift, so ranking performance may differ in future periods.

Furthermore, ranking metrics measure observed subscription outcomes, not the causal effect of contacting customers.

The evaluated contact capacities are illustrative. No final ranking policy is selected at this stage.

Further model comparison and optimization will be performed in `04-model-experiments.ipynb`.


## 5. Baseline Summary

This notebook established a reproducible Logistic Regression baseline for the Bank Marketing classification task.

The selected preprocessing configuration and chronological split were reproduced from the previous notebooks.

The baseline was trained exclusively on the training set and evaluated on the validation set.

### 5.1 Baseline Results

**Model configuration**

- Model: `LogisticRegression(max_iter=1000)`
- Numeric preprocessing: `StandardScaler`, including raw `pdays`
- Categorical preprocessing: `OneHotEncoder(handle_unknown="ignore")`
- Modeling features: 19 predictors, excluding `duration`
- Data split: chronological 70% / 20% / 10%

**General ranking performance**

| Metric | Validation Result |
|---|---:|
| Average Precision | 0.297231 |
| ROC-AUC | 0.692923 |

These results exactly reproduce the selected Logistic Regression baseline from `02-preprocessing.ipynb`.

**Business-oriented ranking performance**

| Contact Capacity | Customers | Captured Subscribers | Precision@K | Recall@K | Lift@K |
|---|---:|---:|---:|---:|---:|
| Top 5% | 412 | 177 | 0.4296 | 0.1550 | 3.0991 |
| Top 10% | 824 | 318 | 0.3859 | 0.2785 | 2.7839 |
| Top 20% | 1,648 | 503 | 0.3052 | 0.4405 | 2.2017 |

The baseline demonstrates an ability to concentrate actual subscribers among higher-ranked customers.

For example, the top 10% of customers contains approximately 27.85% of all actual subscribers in the validation set, with a subscription rate approximately 2.78 times the overall validation positive rate.

These results establish the reference performance for subsequent model experiments.

### 5.2 Limitations

The current baseline has several limitations.

**Model complexity**

Logistic Regression learns a linear decision function in the transformed feature space. It may not adequately capture nonlinear relationships or complex feature interactions.

**Temporal distribution shift**

The dataset exhibits substantial changes in target prevalence and macroeconomic conditions over time.

Validation performance may therefore differ from performance in later periods.

**Validation-specific performance**

The reported metrics describe performance on the established chronological validation period and do not guarantee equivalent performance on future data.

**Business interpretation**

Precision@K, Recall@K, and Lift@K measure ranking effectiveness under illustrative contact capacities.

They do not establish the causal impact of marketing contacts or determine the optimal contact policy.

**Probability calibration**

The model's predicted probabilities have not been evaluated for calibration. Their reliability as absolute subscription probabilities remains unverified.

### 5.3 Next Experiments

The next notebook, `04-model-experiments.ipynb`, will investigate whether alternative modeling approaches can improve validation performance.

Planned experiments include:

1. Compare Logistic Regression with tree-based models, such as Random Forest and Gradient Boosting.
2. Explore model-specific preprocessing and feature representations when justified.
3. Perform targeted hyperparameter optimization for promising models.
4. Analyze model performance, interpretability, and robustness.
5. Select the final model configuration based on validation results.

Average Precision will remain the primary model-selection metric.

ROC-AUC and business-oriented ranking metrics will provide additional evidence about model performance.

All experiments will preserve the established chronological split and prediction-time feature availability assumptions.

The test set remains frozen for final evaluation in `05-final-evaluation.ipynb`.
